# MoMo-FDVS logical PR16 — private Ghana dataset pilot

This owner-operated notebook validates the private intake pipeline against files already placed in restricted Drive storage. It never downloads web content, prints raw messages or images, trains a model, or opens a locked test partition.

In [ ]:
RUN_PROFILE = "smoke"  # governance/pipeline execution only
TARGET_COMMIT = "REPLACE_WITH_PUSHED_PR16_SHA"
REPOSITORY_URL = "https://github.com/davidagyekum/momo-fraud-detection.git"
DRIVE_ROOT = "/content/drive/MyDrive/momo-fraud"
VM_ROOT = "/content/momo-work"
NOTEBOOK_PATH = "ml/notebooks/colab/05_build_ghana_screenshot_dataset.ipynb"
INTAKE_MODE = "screenshots"  # screenshots or messages
assert RUN_PROFILE == "smoke"
assert INTAKE_MODE in {"screenshots", "messages"}


In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive", force_remount=True, timeout_ms=600000)
repo = Path(VM_ROOT) / "repo"
repo.parent.mkdir(parents=True, exist_ok=True)
if (repo / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo), "fetch", "--prune", "origin"], check=True)
else:
    subprocess.run(["git", "clone", "--no-checkout", REPOSITORY_URL, str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", TARGET_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--requirement", str(repo / "ml/requirements-runtime.lock")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--editable", str(repo / "ml")], check=True)
sys.path.insert(0, str(repo / "ml/src"))


In [ ]:
import json
from momo_fdvs_ml.colab import ColabPaths, colab_preflight_report
from momo_fdvs_ml.execution import ExecutionProfile

paths = ColabPaths(drive_root=Path(DRIVE_ROOT), vm_root=Path(VM_ROOT))
preflight = colab_preflight_report(repo, paths=paths, profile=ExecutionProfile.SMOKE, notebook=NOTEBOOK_PATH, require_colab=True)
assert preflight["git"]["commit"] == TARGET_COMMIT
assert preflight["git"]["dirty"] is False
assert preflight["full_training_executed"] is False
print(json.dumps({"commit": TARGET_COMMIT, "profile": RUN_PROFILE, "intake_mode": INTAKE_MODE, "training_executed": False}, indent=2, sort_keys=True))


In [ ]:
from momo_fdvs_ml.ghana_pipeline import index_imazing_messages, ingest_private_screenshots

private_root = Path(DRIVE_ROOT) / "private-governance/ghana-private"
private_root.mkdir(parents=True, exist_ok=True)
if INTAKE_MODE == "screenshots":
    outputs = ingest_private_screenshots(request_path=private_root / "intake-request.json", raw_root=private_root / "raw", working_root=private_root / "working", index_path=private_root / "private-index.json", report_path=private_root / "intake-report.json", repository_root=repo)
else:
    private_config = json.loads((private_root / "message-intake-config.json").read_text(encoding="utf-8"))
    outputs = index_imazing_messages(source_csv=private_root / "owner-messages.csv", index_path=private_root / "message-index.json", report_path=private_root / "message-report.json", repository_root=repo, participant_id_hash=private_config["participant_id_hash"], permission_reference=private_config["permission_reference"])
safe_report = json.loads(outputs.report_path.read_text(encoding="utf-8"))
assert safe_report["training_executed"] is False
print(json.dumps(safe_report, indent=2, sort_keys=True))


## Stop boundary

Stop after the safe aggregate report. Do not print or commit the private index, raw messages, images, consent records or direct identifiers. Do not scrape websites, approve unreviewed online candidates, freeze a test split before the pilot review, train a model, or begin PR17.